# 04 — Optimisation des Hyperparamètres
**Projet** : ObRail MSPR 2025-2026  
**Auteure** : Charlotte  
**Source** : `data/processed/X_train.csv`, `X_test.csv`, `y_train.csv`, `y_test.csv`  

**Objectif** : Optimiser les hyperparamètres de LightGBM (modèle sélectionné en `03_models.ipynb`) via une approche en deux étapes :
1. **RandomizedSearchCV** — explore rapidement un large espace d'hyperparamètres
2. **GridSearchCV** — affine la recherche autour de la meilleure région trouvée

**Justification de l'approche deux étapes** : GridSearchCV seul sur un grand espace serait trop lent (combinaisons exponentielles). RandomizedSearchCV seul manque de précision. La combinaison des deux est le meilleur compromis vitesse/précision.

**Métrique d'optimisation** : F1-score (classe 1 — routes sous-desservies)  
**Validation** : Stratified K-Fold 5 splits  
**Le jeu de test reste intouché** jusqu'à l'évaluation finale.

## 0. Imports et configuration

In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')
import json

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from lightgbm import LGBMClassifier

from sklearn.model_selection import (
    RandomizedSearchCV,
    GridSearchCV,
    StratifiedKFold,
    cross_val_score,
)
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
    roc_auc_score,
    roc_curve,
)

SEED = 42
np.random.seed(SEED)


def find_project_root() -> Path:
    cwd = Path.cwd().resolve()
    for folder in [cwd] + list(cwd.parents):
        if (folder / 'data').exists() and (folder / 'src').exists() and (folder / 'models').exists():
            return folder
    raise FileNotFoundError('Could not find project root from current working directory')

ROOT = find_project_root()
PROCESSED_DIR = ROOT / 'data' / 'processed'
MODELS_DIR = ROOT / 'models'
PLOT_DIR = ROOT / 'evaluation' / 'plots'
MODELS_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

print('✅ Imports OK')

## 1. Chargement des données

In [ ]:
X_train = pd.read_csv(PROCESSED_DIR / 'X_train.csv')
X_test  = pd.read_csv(PROCESSED_DIR / 'X_test.csv')
y_train = pd.read_csv(PROCESSED_DIR / 'y_train.csv').squeeze()
y_test  = pd.read_csv(PROCESSED_DIR / 'y_test.csv').squeeze()

neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
scale_pos_weight = neg / pos

print(f'X_train : {X_train.shape}')
print(f'X_test  : {X_test.shape}')
print(f'scale_pos_weight : {scale_pos_weight:.2f}')

## 2. Baseline — LightGBM avec hyperparamètres par défaut

On rappelle les performances du modèle non optimisé (notebook 03) pour mesurer le gain apporté par l'optimisation.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

baseline_model = LGBMClassifier(
    scale_pos_weight=scale_pos_weight,
    n_estimators=200,
    random_state=SEED,
    n_jobs=1,
    verbose=-1,
)

baseline_model.fit(X_train, y_train)
y_pred_baseline = baseline_model.predict(X_test)

baseline_f1  = f1_score(y_test, y_pred_baseline)
baseline_auc = roc_auc_score(y_test, baseline_model.predict_proba(X_test)[:, 1])

print('Baseline LightGBM (hyperparamètres par défaut) :')
print(f'  Test F1      : {baseline_f1:.4f}')
print(f'  Test ROC-AUC : {baseline_auc:.4f}')
print('\n→ C\'est la référence. L\'optimisation doit améliorer ce score.')

## 3. Étape 1 — RandomizedSearchCV

**Pourquoi RandomizedSearch en premier ?**  
L'espace des hyperparamètres LightGBM est large. Tester toutes les combinaisons (GridSearch) prendrait des heures. RandomizedSearch tire aléatoirement `n_iter=40` combinaisons dans cet espace et identifie la région la plus prometteuse. C'est rapide et couvre bien l'espace.

**Hyperparamètres explorés et leur rôle** :
- `n_estimators` : nombre d'arbres — plus d'arbres = plus précis mais plus lent
- `learning_rate` : vitesse d'apprentissage — petit = plus précis mais nécessite plus d'arbres
- `max_depth` : profondeur maximale des arbres — contrôle la complexité
- `num_leaves` : nombre de feuilles par arbre — plus = plus expressif mais risque d'overfitting
- `min_child_samples` : nombre minimum d'exemples par feuille — régularisation
- `subsample` : fraction des données utilisées par arbre — réduit l'overfitting
- `colsample_bytree` : fraction des features utilisées par arbre — réduit l'overfitting

In [ ]:
param_dist = {
    'n_estimators':      [100, 200, 300, 500],
    'learning_rate':     [0.01, 0.05, 0.1, 0.2],
    'max_depth':         [-1, 5, 7, 10, 15],
    'num_leaves':        [20, 31, 50, 70, 100],
    'min_child_samples': [10, 20, 30, 50],
    'subsample':         [0.6, 0.7, 0.8, 0.9, 1.0],
    'colsample_bytree':  [0.6, 0.7, 0.8, 0.9, 1.0],
}

lgbm_base = LGBMClassifier(
    scale_pos_weight=scale_pos_weight,
    random_state=SEED,
    n_jobs=1,
    verbose=-1,
)

random_search = RandomizedSearchCV(
    estimator=lgbm_base,
    param_distributions=param_dist,
    n_iter=40,
    scoring='f1',
    cv=cv,
    random_state=SEED,
    n_jobs=1,
    verbose=1,
)

print('Lancement RandomizedSearchCV (40 combinaisons × 5 folds = 200 fits)...')
random_search.fit(X_train, y_train)

print(f'\n✅ RandomizedSearch terminé')
print(f'Meilleur score CV F1 : {random_search.best_score_:.4f}')
print(f'Meilleurs hyperparamètres :')
for k, v in random_search.best_params_.items():
    print(f'  {k}: {v}')

In [ ]:
# Évaluation du meilleur modèle RandomizedSearch sur le test
best_random = random_search.best_estimator_
y_pred_random = best_random.predict(X_test)

random_f1  = f1_score(y_test, y_pred_random)
random_auc = roc_auc_score(y_test, best_random.predict_proba(X_test)[:, 1])

print(f'RandomizedSearch — résultats sur test :')
print(f'  Test F1      : {random_f1:.4f}  (baseline : {baseline_f1:.4f})')
print(f'  Test ROC-AUC : {random_auc:.4f}  (baseline : {baseline_auc:.4f})')
print(f'  Gain F1      : +{random_f1 - baseline_f1:.4f}')

## 4. Étape 2 — GridSearchCV

**Pourquoi GridSearch en second ?**  
RandomizedSearch a identifié la région prometteuse. On construit maintenant une grille fine autour des meilleures valeurs trouvées pour affiner précisément. La grille est petite donc GridSearch reste rapide.

On prend les valeurs trouvées par RandomizedSearch et on teste les valeurs voisines.

In [ ]:
best_p = random_search.best_params_

def neighbors(val, options):
    """Retourne val et ses voisins dans la liste options."""
    if val not in options:
        return [val]
    idx = options.index(val)
    return list(set(options[max(0, idx-1):idx+2]))

n_est_opts     = [100, 200, 300, 500]
lr_opts        = [0.01, 0.05, 0.1, 0.2]
depth_opts     = [-1, 5, 7, 10, 15]
leaves_opts    = [20, 31, 50, 70, 100]
child_opts     = [10, 20, 30, 50]
sub_opts       = [0.6, 0.7, 0.8, 0.9, 1.0]
col_opts       = [0.6, 0.7, 0.8, 0.9, 1.0]

param_grid = {
    'n_estimators':      neighbors(best_p['n_estimators'],      n_est_opts),
    'learning_rate':     neighbors(best_p['learning_rate'],     lr_opts),
    'max_depth':         neighbors(best_p['max_depth'],         depth_opts),
    'num_leaves':        neighbors(best_p['num_leaves'],        leaves_opts),
    'min_child_samples': neighbors(best_p['min_child_samples'], child_opts),
    'subsample':         neighbors(best_p['subsample'],         sub_opts),
    'colsample_bytree':  neighbors(best_p['colsample_bytree'],  col_opts),
}

total = 1
for v in param_grid.values():
    total *= len(v)
print(f'Grille GridSearch : {total} combinaisons × 5 folds = {total*5} fits')
print('Paramètres testés :')
for k, v in param_grid.items():
    print(f'  {k}: {v}')

In [ ]:
lgbm_grid = LGBMClassifier(
    scale_pos_weight=scale_pos_weight,
    random_state=SEED,
    n_jobs=1,
    verbose=-1,
)

grid_search = GridSearchCV(
    estimator=lgbm_grid,
    param_grid=param_grid,
    scoring='f1',
    cv=cv,
    n_jobs=1,
    verbose=1,
)

print('Lancement GridSearchCV...')
grid_search.fit(X_train, y_train)

print(f'\n✅ GridSearch terminé')
print(f'Meilleur score CV F1 : {grid_search.best_score_:.4f}')
print(f'Meilleurs hyperparamètres :')
for k, v in grid_search.best_params_.items():
    print(f'  {k}: {v}')

In [ ]:
# Évaluation du meilleur modèle GridSearch sur le test
best_grid = grid_search.best_estimator_
y_pred_grid = best_grid.predict(X_test)

grid_f1  = f1_score(y_test, y_pred_grid)
grid_auc = roc_auc_score(y_test, best_grid.predict_proba(X_test)[:, 1])

print(f'GridSearch — résultats sur test :')
print(f'  Test F1      : {grid_f1:.4f}  (baseline : {baseline_f1:.4f})')
print(f'  Test ROC-AUC : {grid_auc:.4f}  (baseline : {baseline_auc:.4f})')
print(f'  Gain F1 vs baseline       : +{grid_f1 - baseline_f1:.4f}')
print(f'  Gain F1 vs RandomizedSearch : +{grid_f1 - random_f1:.4f}')

## 5. Tableau comparatif — baseline vs RandomizedSearch vs GridSearch

In [ ]:
optim_comparison = pd.DataFrame([
    {
        'Étape':         'Baseline (défaut)',
        'Test F1':       round(baseline_f1, 4),
        'Test ROC-AUC':  round(baseline_auc, 4),
        'Description':   'n_estimators=200, hyperparamètres par défaut',
    },
    {
        'Étape':         'RandomizedSearch',
        'Test F1':       round(random_f1, 4),
        'Test ROC-AUC':  round(random_auc, 4),
        'Description':   '40 combinaisons aléatoires, large espace de recherche',
    },
    {
        'Étape':         'GridSearch (affinement)',
        'Test F1':       round(grid_f1, 4),
        'Test ROC-AUC':  round(grid_auc, 4),
        'Description':   'Grille fine autour des meilleurs paramètres RandomizedSearch',
    },
])

print('Comparaison des étapes d\'optimisation :')
display(optim_comparison)

optim_comparison.to_csv('../evaluation/optimization_comparison.csv', index=False)
print('\n✅ Sauvegardé → evaluation/optimization_comparison.csv')

## 6. Évaluation finale du modèle optimisé

In [ ]:
# Le modèle final est celui avec le meilleur F1 entre RandomizedSearch et GridSearch
if grid_f1 >= random_f1:
    final_model = best_grid
    final_name  = 'LightGBM (GridSearch)'
    final_params = grid_search.best_params_
else:
    final_model = best_random
    final_name  = 'LightGBM (RandomizedSearch)'
    final_params = random_search.best_params_

y_pred_final = final_model.predict(X_test)
y_proba_final = final_model.predict_proba(X_test)[:, 1]

final_f1  = f1_score(y_test, y_pred_final)
final_auc = roc_auc_score(y_test, y_proba_final)

print(f'✅ Modèle final : {final_name}')
print(f'   Test F1      : {final_f1:.4f}')
print(f'   Test ROC-AUC : {final_auc:.4f}')
print(f'\nClassification report :')
print(classification_report(
    y_test, y_pred_final,
    target_names=['Non sous-desservi', 'Sous-desservi']
))

In [ ]:
# Matrice de confusion du modèle final
fig, ax = plt.subplots(figsize=(6, 5))
cm = confusion_matrix(y_test, y_pred_final)
ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=['Non sous-desservi', 'Sous-desservi']
).plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title(f'Matrice de confusion — {final_name}')
plt.tight_layout()
plt.savefig(PLOT_DIR / 'confusion_matrix_final.png')
plt.show()

In [ ]:
# Courbe ROC — comparaison baseline vs modèle optimisé
fig, ax = plt.subplots(figsize=(8, 6))

for model, label, color in [
    (baseline_model, f'Baseline (F1={baseline_f1:.3f})', '#95a5a6'),
    (final_model,    f'{final_name} (F1={final_f1:.3f})', '#e74c3c'),
]:
    fpr, tpr, _ = roc_curve(y_test, model.predict_proba(X_test)[:, 1])
    ax.plot(fpr, tpr, label=label, linewidth=2, color=color)

ax.plot([0, 1], [0, 1], 'k--', label='Aléatoire')
ax.set_xlabel('Taux de faux positifs')
ax.set_ylabel('Taux de vrais positifs')
ax.set_title('Courbe ROC — Baseline vs Modèle optimisé')
ax.legend()
plt.tight_layout()
plt.savefig(PLOT_DIR / 'roc_curve_optimized.png')
plt.show()

## 7. Sauvegarde du modèle final optimisé

In [ ]:
# Sauvegarde du modèle
final_model_path = MODELS_DIR / 'best_model_optimized.joblib'
joblib.dump(final_model, final_model_path)

# Mise à jour des métadonnées
metadata = {
    'model_name':        final_name,
    'test_f1':           round(final_f1, 4),
    'test_roc_auc':      round(final_auc, 4),
    'baseline_f1':       round(baseline_f1, 4),
    'gain_f1':           round(final_f1 - baseline_f1, 4),
    'best_params':       final_params,
    'features':          list(X_train.columns),
    'target':            'is_underserved',
    'n_train':           len(X_train),
    'n_test':            len(X_test),
    'optimization':      'RandomizedSearchCV (40 iter) → GridSearchCV (fine-tuning)',
    'cv_strategy':       'StratifiedKFold(n_splits=5)',
    'scoring_metric':    'f1 (classe 1 — routes sous-desservies)',
    'seed':              SEED,
}

with open(MODELS_DIR / 'model_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

print(f'✅ Modèle optimisé sauvegardé → {final_model_path}')
print(f'✅ Métadonnées mises à jour → models/model_metadata.json')
print(f'\nRésumé :')
print(f'  Baseline F1  : {baseline_f1:.4f}')
print(f'  Optimisé F1  : {final_f1:.4f}')
print(f'  Gain         : +{final_f1 - baseline_f1:.4f}')

## 8. Récapitulatif

**Approche d'optimisation**
- Étape 1 : RandomizedSearchCV — 40 combinaisons aléatoires sur large espace → identifie la région optimale rapidement
- Étape 2 : GridSearchCV — grille fine autour des meilleurs paramètres → affine précisément
- Justification : meilleur compromis vitesse/précision vs GridSearch seul (trop lent) ou RandomizedSearch seul (moins précis)

**Résultats**
- Baseline F1 : résultats notebook 03
- Modèle optimisé F1 : voir cellule 5
- Le modèle final est sauvegardé → `models/best_model_optimized.joblib`
- Les métadonnées complètes (features, params, scores) sont dans `models/model_metadata.json`

**Prochaines étapes**
- `notebooks/05_explainability.ipynb` — SHAP et LIME sur le modèle optimisé
- `src/predict.py` — script de prédiction utilisant `best_model_optimized.joblib`
- `api/main.py` — API FastAPI `/predict`